### Riemann Sum Integration GUI
- Enter an integrand function (in terms of `x`, using `np.` for numpy functions), lower bound, upper bound, and number of rectangles
- Click **Compute** to plot the function, draw the Riemann rectangles, and show the approximate integral
- Uses `ipywidgets` for the interactive GUI

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [3]:
# --- Widgets for user input ---
func_text = widgets.Text(
    value='x**2 + 3*x + 5',
    description='f(x) =',
    layout=widgets.Layout(width='400px')
)
lb_box = widgets.FloatText(value=1, description='Lower bound')
ub_box = widgets.FloatText(value=8, description='Upper bound')
n_box = widgets.IntSlider(value=10, min=1, max=200, step=1, description='N rectangles')
method_box = widgets.Dropdown(
    options=[('Left', 'left'), ('Right', 'right'), ('Midpoint', 'mid')],
    value='right',
    description='Method'
)
compute_btn = widgets.Button(description='Compute', button_style='primary')
output = widgets.Output()


def make_function(expr):
    """Turn a user-typed string expression of x into a callable f(x)."""
    # only expose numpy and x to keep eval scope limited
    allowed_names = {'np': np, 'x': None}

    def f(x):
        return eval(expr, {'__builtins__': {}}, {'np': np, 'x': x})
    return f


def on_compute_clicked(b):
    output.clear_output(wait=True)
    with output:
        try:
            f = make_function(func_text.value)
            lb, ub = lb_box.value, ub_box.value
            N = n_box.value
            method = method_box.value

            if ub <= lb:
                print('Upper bound must be greater than lower bound.')
                return

            # rectangle edges and sample points based on chosen method
            edges = np.linspace(lb, ub, N + 1)
            width = edges[1:] - edges[:-1]
            if method == 'left':
                sample = edges[:-1]
            elif method == 'right':
                sample = edges[1:]
            else:
                sample = (edges[:-1] + edges[1:]) / 2

            heights = f(sample)
            riemann_sum = np.sum(heights * width)

            # plot the function over a slightly wider range for context
            margin = (ub - lb) * 0.2 if ub > lb else 1
            x_plot = np.linspace(lb - margin, ub + margin, 300)
            y_plot = f(x_plot)

            fig, ax = plt.subplots(figsize=(7, 5))
            ax.plot(x_plot, y_plot, color='black', label='f(x)')

            for i in range(N):
                ax.fill_between([edges[i], edges[i + 1]], [heights[i], heights[i]], 0,
                                 step=None, alpha=0.4, color='b', edgecolor='k', linewidth=0.5)

            ax.axhline(0, color='gray', linewidth=0.8)
            ax.set_xlabel('x')
            ax.set_ylabel('f(x)')
            ax.set_title(f'Riemann Sum ({method}) with N={N}: Approx. Integral = {riemann_sum:.4f}')
            ax.legend()
            ax.grid(True, linestyle='--', which='major')
            plt.show()

            print(f'Approximate integral (Riemann sum) = {riemann_sum:.6f}')
        except Exception as e:
            print(f'Error evaluating function: {e}')


compute_btn.on_click(on_compute_clicked)

ui = widgets.VBox([
    func_text,
    widgets.HBox([lb_box, ub_box]),
    n_box,
    method_box,
    compute_btn,
    output
])
display(ui)

In [4]:
import matplotlib.animation as animation
from IPython.display import HTML

# --- Widgets for the animated version (separate from the static GUI above) ---
anim_func_text = widgets.Text(
    value='x**2 + 3*x + 5',
    description='f(x) =',
    layout=widgets.Layout(width='300px')
)
anim_lb_box = widgets.FloatText(value=1, description='Lower bound', layout=widgets.Layout(width='200px'))
anim_ub_box = widgets.FloatText(value=8, description='Upper bound', layout=widgets.Layout(width='200px'))
anim_n_box = widgets.IntSlider(value=10, min=1, max=100, step=1, description='N rectangles')
anim_method_box = widgets.Dropdown(
    options=[('Left', 'left'), ('Right', 'right'), ('Midpoint', 'mid')],
    value='right',
    description='Method',
    layout=widgets.Layout(width='200px')
)
anim_speed_box = widgets.IntSlider(value=300, min=50, max=1000, step=50, description='ms/frame')
anim_btn = widgets.Button(description='Animate', button_style='primary')
anim_output = widgets.Output()


def on_animate_clicked(b):
    anim_output.clear_output(wait=True)
    with anim_output:
        try:
            f = make_function(anim_func_text.value)
            lb, ub = anim_lb_box.value, anim_ub_box.value
            N = anim_n_box.value
            method = anim_method_box.value

            if ub <= lb:
                print('Upper bound must be greater than lower bound.')
                return

            edges = np.linspace(lb, ub, N + 1)
            width = edges[1:] - edges[:-1]
            if method == 'left':
                sample = edges[:-1]
            elif method == 'right':
                sample = edges[1:]
            else:
                sample = (edges[:-1] + edges[1:]) / 2

            heights = f(sample)

            margin = (ub - lb) * 0.2 if ub > lb else 1
            x_plot = np.linspace(lb - margin, ub + margin, 300)
            y_plot = f(x_plot)

            fig, ax = plt.subplots(figsize=(7, 5))
            ax.plot(x_plot, y_plot, color='black', label='f(x)')
            ax.axhline(0, color='gray', linewidth=0.8)
            ax.set_xlabel('x')
            ax.set_ylabel('f(x)')
            ax.legend()
            ax.grid(True, linestyle='--', which='major')

            # one rectangle patch added and updated per animation frame
            bars = ax.bar(edges[:-1], heights, width=width, align='edge',
                           alpha=0.4, color='b', edgecolor='k', linewidth=0.5)
            for bar in bars:
                bar.set_visible(False)
            title = ax.set_title('')

            def update(frame):
                bars[frame].set_visible(True)
                running_sum = np.sum(heights[:frame + 1] * width[:frame + 1])
                title.set_text(f'Riemann Sum ({method}) N={N}: rectangle {frame + 1}/{N}, '
                                f'partial sum = {running_sum:.4f}')
                return list(bars[:frame + 1]) + [title]

            anim = animation.FuncAnimation(fig, update, frames=N,
                                            interval=anim_speed_box.value,
                                            blit=False, repeat=False)
            plt.close(fig)
            display(HTML(anim.to_jshtml()))

            riemann_sum = np.sum(heights * width)
            print(f'Approximate integral (Riemann sum) = {riemann_sum:.6f}')
        except Exception as e:
            print(f'Error evaluating function: {e}')


anim_btn.on_click(on_animate_clicked)

anim_ui = widgets.VBox([
    anim_func_text,
    widgets.HBox([anim_lb_box, anim_ub_box]),
    anim_n_box,
    anim_method_box,
    anim_speed_box,
    anim_btn,
    anim_output
])
display(anim_ui)
